<a href="https://colab.research.google.com/github/nabtahilrehman/Nabtahil-flyrank-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabtahilrehman/Nabtahil-flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
## Method Choice

'''I selected a Random Forest model for this lane.

The objective is to identify content that may represent a search opportunity using historical search performance signals.

Random Forest is suitable because it can capture non-linear relationships between impressions, clicks, position, and engagement metrics without requiring strong distribution assumptions.

The model is intended as a decision-support tool that prioritizes opportunities rather than making final decisions automatically.'''


'I selected a Random Forest model for this lane.\n\nThe objective is to identify content that may represent a search opportunity using historical search performance signals.\n\nRandom Forest is suitable because it can capture non-linear relationships between impressions, clicks, position, and engagement metrics without requiring strong distribution assumptions.\n\nThe model is intended as a decision-support tool that prioritizes opportunities rather than making final decisions automatically.'

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
## Split Design

'''A time-aware split is used.

Earlier observations are used for training and later observations are used for evaluation.

This approach is more realistic because it mirrors how a model would be used in practice, where future data is unavailable at training time.

The split helps reduce leakage and provides a more honest estimate of performance.'''


'A time-aware split is used.\n\nEarlier observations are used for training and later observations are used for evaluation.\n\nThis approach is more realistic because it mirrors how a model would be used in practice, where future data is unavailable at training time.\n\nThe split helps reduce leakage and provides a more honest estimate of performance.'

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
import os

if not os.path.isdir("flyrank-ml-internship-starter"):
    !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

os.chdir("flyrank-ml-internship-starter")
print("Now in:", os.getcwd())

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 299 (delta 130), reused 98 (delta 98), pack-reused 106 (from 1)
Receiving objects: 100% (299/299), 1.88 MiB | 4.59 MiB/s, done.
Resolving deltas: 100% (161/161), done.
Now in: /content/flyrank-ml-internship-starter


In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["ctr"] = df["ctr"].fillna(0)
df["score"] = df["impressions_90d"] * (1 - df["ctr"])   # same opportunity score as Week 1/4

# Content-intrinsic features only — nothing derived from impressions/clicks/ctr/traffic history,
# since those are the inputs to the target itself
num_features = ["search_volume", "word_count", "content_age_days", "days_since_last_update",
                 "avg_position", "engagement_rate", "scroll_rate"]
cat_features = ["competition_level", "content_type", "main_intent"]

data = df.dropna(subset=num_features + cat_features + ["score", "client_id"]).copy()
X = data[num_features + cat_features]
y = data["score"]
groups = data["client_id"]

# Grouped split by client — the CSV has no date column, so a time-aware split isn't
# possible here; grouping by client is the honest choice, preventing the same
# client's pages from appearing in both train and test.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)],
                         remainder="passthrough")
model = Pipeline([("pre", pre), ("rf", RandomForestRegressor(n_estimators=200, random_state=42))])
model.fit(X_train, y_train)
mae_rf = mean_absolute_error(y_test, model.predict(X_test))
print("Random Forest MAE:", round(mae_rf, 2))

Random Forest MAE: 3865.7


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
'''The model produced an MAE of 3865.7, meaning predictions are off by about 3,866 score units on average. Errors likely occur because only content-related features are used and historical traffic metrics were excluded. The model appears to capture general patterns but may struggle with client-specific behavior and pages with unusually high or low opportunity scores.'''

'The model produced an MAE of 3865.7, meaning predictions are off by about 3,866 score units on average. Errors likely occur because only content-related features are used and historical traffic metrics were excluded. The model appears to capture general patterns but may struggle with client-specific behavior and pages with unusually high or low opportunity scores.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.